In [1]:
import pandas as pd

# 1.1 Učitavanje i čišćenje podataka

In [2]:
sheets = pd.read_excel("../data/podaci.xlsx", sheet_name=None)

print(f"Listovi: {list(sheets.keys())}")
for name, part in sheets.items():
    print(f"  {name}: {len(part):,} redova")

parts = []
for name, part in sheets.items():
    part = part.copy()
    part["List"] = name 
    parts.append(part)

df = pd.concat(parts, ignore_index=True)

Listovi: ['Year 2009-2010', 'Year 2010-2011']
  Year 2009-2010: 525,461 redova
  Year 2010-2011: 541,910 redova


## Keširanje

In [3]:
for col in ["Invoice", "StockCode", "Description"]:
    df[col] = df[col].astype(str).str.strip()

df.to_parquet("../data/sirovo.parquet", index=False)

In [4]:
df = pd.read_parquet("../data/sirovo.parquet")

# 1.2 Podaci

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1067371 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
 8   List         1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 73.3+ MB


In [7]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [8]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,List
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [9]:
df.isna().sum()

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
List                0
dtype: int64

In [10]:
df.nunique()

Invoice        53628
StockCode       5304
Description     5656
Quantity        1057
InvoiceDate    47635
Price           2807
Customer ID     5942
Country           43
List               2
dtype: int64

## Preimenovanje kolona

In [11]:
df = df.rename(columns={
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID"
})
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'List'],
      dtype='object')


**1. Koliko redova i kolona ima skup?**

- **Redova:** 1 067 371
- **Kolona:** 9 (8 originalnih, bez kolone `List`)

**2. Koje kolone imaju nedostajuće vrijednosti i koliko?**

| Kolona       | Nedostajućih vrijednosti |
|--------------|--------------------------|
| `CustomerID` | 243 007                  |

**3. Šta pokazuje `.describe()` za `Quantity` i `Price` (min) i kako objašnjavate negativne vrijednosti?**

- `Quantity` i `Price` imaju **negativan minimum**.
- U podacima su i **povraćaji i korekcije**, ne samo prodaja.
- `max` je ogroman u odnosu na medijanu, što znači da postoje **ekstremne vrijednosti (outlieri)**.

**4. Preimenovanje kolona: zašto je dobro imati jedan dogovoren skup naziva?**

Kolone `Invoice`, `Price`, `Customer ID` preimenovane su u `InvoiceNo`, `UnitPrice`, `CustomerID`.

Treba poštovati konvenciju i biti dosljedan sa standardima, jer:
- kod iz tutorijala radi bez prilagođavanja,
- manje je grešaka zbog različitih naziva,
- kod je čitljiviji i lakši za održavanje.

# 1.3 Preklapanje listova

In [13]:
for name, part in df.groupby("List"):
    print(f"{name}: {part['InvoiceDate'].min()} -> {part['InvoiceDate'].max()}")

Year 2009-2010: 2009-12-01 07:45:00 -> 2010-12-09 20:01:00
Year 2010-2011: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [15]:
invoice_by_sheet = df.groupby("List")["InvoiceNo"].apply(set)
common = set.intersection(*invoice_by_sheet.tolist())

print(f"Računa u OBA lista: {len(common):,}")
print(f"Redova koji im pripadaju: {df['InvoiceNo'].isin(common).sum():,}")

Računa u OBA lista: 1,088
Redova koji im pripadaju: 45,046



**1. Da li se periodi preklapaju? U kom tačno intervalu?**

Da, periodi se preklapaju od **2010-12-01** do **2010-12-09**.

**2. Koliko računa postoji u oba lista?**

**1088** računa postoji u oba lista.

**3. Šta bi se desilo s prikazanom prodajom za decembar 2010. da su redovi ostali duplirani?**

Tih 1088 računa bi bilo upisano **dvaput**. Ako se duplikati ne uklone, prodaja za decembar 2010. bila bi prikazana **većom nego što zaista jeste**.

# 1.4 Čišćenje podataka

In [16]:
steps = []

def cleaning_report(name: str, before: int, after: int):
    removed = before - after 

    d = {
        "Korak": name,
        "Prije": before,
        "Poslije": after,
        "Uklonjeno": removed
    }
    steps.append(d)
    print(f"{name:<34} {before:>9,} -> {after:>9,} (-{removed:,})")

## Duplikati

In [17]:
bussines = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]
before = len(df)
df = df.drop_duplicates(subset=bussines, keep="first").reset_index(drop=True)

cleaning_report("Uklonjeni duplikati", before, len(df))

Uklonjeni duplikati                1,067,371 -> 1,033,036 (-34,335)


## Otkazivanja

In [18]:
df["JeOtkazivanje"] = df["InvoiceNo"].str.upper().str.startswith("C")

cancellations = df[df["JeOtkazivanje"]].copy()
cancellations.to_parquet("../data/otkazivanja.parquet", index=False)

before = len(df)
df = df[~df["JeOtkazivanje"]].copy()
cleaning_report("Izdvojena otkazivanja", before, len(df))

Izdvojena otkazivanja              1,033,036 -> 1,013,932 (-19,104)


## CustomerID redovi

In [19]:
before = len(df)
df = df[df["CustomerID"].notna()].copy()
cleaning_report("Uklonjeni redovi bez CustomerID", before, len(df))

Uklonjeni redovi bez CustomerID    1,013,932 ->   779,495 (-234,437)


## Službene šifre

In [20]:
print(df.StockCode.value_counts().head(20))

StockCode
85123A    5023
22423     3337
85099B    3296
84879     2692
20725     2609
21212     2557
47566     2099
20727     2045
22383     2039
21034     1950
22382     1935
21232     1935
22384     1874
21754     1852
22139     1848
20914     1822
22469     1821
20728     1820
84991     1813
POST      1803
Name: count, dtype: int64


In [ ]:
codes = {
    "POST",
    "DOT",
    "C2",
    "C3",
    "M",
    "D",
    "S",
    "BANK CHARGES",
    "ADJUST",
    "ADJUST2",
    "AMAZONFEE",
    "CRUK",
    "PADS",
    "TEST001",
    "TEST002",
}

up = df.StockCode.str.upper().str.strip()
mask = up.isin(codes) | up.str.startswith("GIFT_0001")

before = len(df)
df = df[~mask].copy()
cleaning_report("Uklonjene sluzbene sifre", before, len(df))

Uklonjene sluzbene sifre             779,495 ->   776,639 (-2,856)


## Quantity, UnitPrice <= 0

In [23]:
before = len(df)
df = df[df["Quantity"] > 0].copy()
cleaning_report("Uklonjeno Quantity <= 0", before, len(df))

before = len(df)
df = df[df["UnitPrice"] > 0].copy()
cleaning_report("Uklonjeno UnitPrice <= 0", before, len(df))

Uklonjeno Quantity <= 0              776,639 ->   776,639 (-0)
Uklonjeno UnitPrice <= 0             776,639 ->   776,579 (-60)


## Kolone

In [26]:
df["CustomerID"] = df["CustomerID"].astype("int64")
df["TotalPrice"] = (df["Quantity"] * df["UnitPrice"]).round(2)

df["Godina"] = df["InvoiceDate"].dt.year
df["Mesec"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["Datum"] = df["InvoiceDate"].dt.date
df["DanUNedelji"] = df["InvoiceDate"].dt.dayofweek
df["Sat"] = df["InvoiceDate"].dt.hour

df = df.drop(columns=["JeOtkazivanje"])

## Provjera

In [27]:
print(f"Redova           : {len(df):,}")
print(f"Kupaca           : {df['CustomerID'].nunique():,}")
print(f"Racuna           : {df['InvoiceNo'].nunique():,}")
print(f"Artikala         : {df['StockCode'].nunique():,}")
print(f"Ukupan prihod    : {df['TotalPrice'].sum():,.0f}")
print(f"Nema CustomerID  : {df['CustomerID'].isna().sum()}")
print(f"Quantity <= 0    : {(df['Quantity'] <= 0).sum()}")

Redova           : 776,579
Kupaca           : 5,852
Racuna           : 36,594
Artikala         : 4,620
Ukupan prihod    : 17,068,583
Nema CustomerID  : 0
Quantity <= 0    : 0


## Čisti podaci

In [28]:
df.to_parquet("../data/cisto.parquet", index=False)

**1. Koliko procenata podataka ste izbacili? Da li je to previše?**

Izbačeno je **27,2%** podataka.

**2. Zašto redove bez `CustomerID` izbacujemo za analizu kupaca, a ne iz ukupne prodaje?**

- **Analiza kupaca:** bez šifre kupca ne može se izračunati koliko je puta taj kupac kupovao, pa ti redovi ne koriste.
- **Ukupna prodaja:** roba je stvarno prodana, pa ti redovi ostaju u ukupnoj prodaji.

**3. Koliko je redova s `Quantity <= 0` ostalo nakon uklanjanja otkazivanja? Šta to govori o negativnim količinama?**

**Nijedan.** Sve negativne količine bile su u otkazanim računima, koji su već izdvojeni.